#Step 1: Data Hygiene

I am going to be doing a hygiene pass, identifying errors, and creating a validation ruleset to guide future hygiene efforts. Then, using the validation rules, I will make a set of cleaned tables to be used in our funnel analysis and workflow design.

###Summary

I created clean versions of each CRM table, applying standardized transformations and validation flags to track known data quality issues for audit and downstream use.

Duplicate records were identified and flagged, but not removed, in order to preserve raw data and enable controlled deduplication logic:

- Companies: 110 duplicates (11%)
- Contacts: 687 duplicates (10.5%)
- Campaign Touchpoints & Deals: No duplicates identified

In addition to deduplication, I implemented a set of Data Governance & Cleaning Rules to standardize key fields (e.g., country, industry, job titles) and to flag issues such as invalid relationships, lifecycle inconsistencies, and timestamp anomalies.

These rules establish a foundation for reliable reporting and will be leveraged in later steps to power workflow automation, lead scoring, and lifecycle management.

### Section 1: Initial Data Audit

This is an initial pass of each table to outline the shape, value uniqueness, and missing values

In [0]:
select
  *
from information_schema.columns
where table_name in ('crm_raw_contacts','crm_raw_companies','crm_raw_deals','crm_raw_campaign_touchpoints')

table_catalog,table_schema,table_name,column_name,ordinal_position,column_default,is_nullable,full_data_type,data_type,character_maximum_length,character_octet_length,numeric_precision,numeric_precision_radix,numeric_scale,datetime_precision,interval_type,interval_precision,maximum_cardinality,is_identity,identity_generation,identity_start,identity_increment,identity_maximum,identity_minimum,identity_cycle,is_generated,generation_expression,is_system_time_period_start,is_system_time_period_end,system_time_period_timestamp_generation,is_updatable,partition_index,comment
workspace,default,crm_raw_campaign_touchpoints,touchpoint_id,0,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_campaign_touchpoints,contact_id,1,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_campaign_touchpoints,campaign_name,2,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_campaign_touchpoints,campaign_type,3,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_campaign_touchpoints,touchpoint_date,4,null,YES,timestamp,TIMESTAMP,null,null,null,null,null,3,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_campaign_touchpoints,touchpoint_action,5,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_campaign_touchpoints,utm_source,6,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_campaign_touchpoints,high_intent_flag,7,null,YES,boolean,BOOLEAN,null,null,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_companies,company_id,0,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_companies,company_name,1,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null


In [0]:
select
  count(*) as total_rows,
  count(distinct contact_id) as total_ids,
  sum(case when company_id is null then 1 else 0 end) as missing_company,
  sum(case when company_id is null then 1 else 0 end) / count(*) * 100 as pct_empty_company_id,
  sum(case when email is null then 1 else 0 end) / count(*) * 100 as pct_empty_email,
  sum(case when email not like '%_@__%.__%' then 1 else 0 end) / count(*) * 100 as pct_invalid_email,
  (count(email) - count(distinct email)) / count(email) * 100 as pct_duplicate_email,
  sum(case when job_title is null then 1 else 0 end) as missing_job_title,
  sum(case when job_title is null then 1 else 0 end) / count(*) * 100 as pct_empty_job_title
from crm_raw_contacts

total_rows,total_ids,missing_company,pct_empty_company_id,pct_empty_email,pct_invalid_email,pct_duplicate_email,missing_job_title,pct_empty_job_title
6500,6500,1625,25.0,0.0,5.353846153846154,6.446153846153846,650,10.0


In [0]:
select
  sum(case when lifecycle_stage = 'Subscriber' then 1 else 0 end) / count(*) * 100 as pct_Subscriber,
  sum(case when lifecycle_stage = 'Lead' then 1 else 0 end) / count(*) * 100 as pct_Lead,
  sum(case when lifecycle_stage = 'Opportunity' then 1 else 0 end) / count(*) * 100 as pct_Opportunity,
  sum(case when lifecycle_stage = 'MQL' then 1 else 0 end) / count(*) * 100 as pct_MQL,
  sum(case when lifecycle_stage = 'SQL' then 1 else 0 end) / count(*) * 100 as pct_SQL,
  sum(case when lifecycle_stage = 'Disqualified' then 1 else 0 end) / count(*) * 100 as pct_Disqualified,
  sum(case when lifecycle_stage = 'Customer' then 1 else 0 end) / count(*) * 100 as pct_Customer
from crm_raw_contacts

pct_Subscriber,pct_Lead,pct_Opportunity,pct_MQL,pct_SQL,pct_Disqualified,pct_Customer
15.03076923076923,36.96923076923077,7.415384615384616,19.876923076923077,10.276923076923076,7.36923076923077,3.0615384615384613


In [0]:
select
  email,
  count(*) as count
from crm_raw_contacts
where email is not null
group by email
having count(*) > 1

email,count
olivia.rodriguez@yahoo.com,2
sophia.white@terradevelopment74.energy,3
nina.green@auroraholdings34.net,2
drew.thompson@prairieanalytics90.io,3
liam.davis@blueridgepartners40.net,2
reese.green@summitdevelopment86.net,2
bad_email,127
parker.young@keystoneinfrastruc84.net,2
sam.anderson@ironwoodpower30.energy,2
blake.smith@ironwoodcapital41.net,2


In [0]:
select
  count(*) as total_rows,
  count(distinct company_id) as total_ids,
  sum(case when domain is null then 1 else 0 end) / count(*) * 100 as pct_empty_domain,
  (count(domain) - count(distinct domain)) / count(domain) * 100 as pct_duplicate_domain,
  sum(case when industry is null then 1 else 0 end) / count(*) * 100 as pct_empty_industry,
  sum(case when company_size_band is null then 1 else 0 end) / count(*) * 100 as pct_empty_company_size_band
from crm_raw_companies

total_rows,total_ids,pct_empty_domain,pct_duplicate_domain,pct_empty_industry,pct_empty_company_size_band
1000,1000,10.0,6.222222222222222,22.0,24.0


In [0]:
select
  domain,
  count(*) as count
from crm_raw_companies
where domain is not null
group by domain
having count(*) > 1

domain,count
riverstoneinfrastr65.io,2
voltworks93.io,3
auroraworks34.com,2
prairieindustries70.capital,2
gridlinefinance45.net,2
blueridgefinance23.com,2
keystoneinfrastruc81.energy,2
prairiesolutions49.capital,2
voltgrid46.energy,2
vertextaxadvisors15.energy,2


In [0]:
select
  count(*) as total_rows,
  count(distinct touchpoint_id) as total_ids,
  count(distinct contact_id) as total_unique_contacts,
  count(distinct contact_id) / count(contact_id) * 100 as pct_unique_contacts
from crm_raw_campaign_touchpoints

total_rows,total_ids,total_unique_contacts,pct_unique_contacts
15000,15000,3258,21.72


In [0]:
select
  count(*) as total_rows,
  count(distinct deal_id) as total_ids,
  count(distinct company_id) as total_unique_companies,
  count(distinct company_id) / count(company_id) * 100 as pct_unique_companies,
  sum(case when company_id is null then 1 else 0 end) / count(*) * 100 as pct_empty_companies,
  count(distinct primary_contact_id) as total_unique_contacts,
  count(distinct primary_contact_id) / count(primary_contact_id) * 100 as pct_unique_contacts,
  sum(case when primary_contact_id is null then 1 else 0 end) / count(*) * 100 as pct_empty_contacts,
  sum(case when deal_stage is null then 1 else 0 end) / count(*) * 100 as pct_deal_stage,
  sum(case when amount is null then 1 else 0 end) / count(*) * 100 as pct_empty_amount,
  sum(case when created_at is null then 1 else 0 end) / count(*) * 100 as pct_empty_created_at,
  sum(case when closed_at is null then 1 else 0 end) / count(*) * 100 as pct_empty_closed_at,
  sum(case when pipeline_source is null then 1 else 0 end) / count(*) * 100 as pct_empty_pipeline_source,
  sum(case when influenced_by_campaign is null then 1 else 0 end) / count(*) * 100 as pct_empty_influenced_by_campaign
from crm_raw_deals

total_rows,total_ids,total_unique_companies,pct_unique_companies,pct_empty_companies,total_unique_contacts,pct_unique_contacts,pct_empty_contacts,pct_deal_stage,pct_empty_amount,pct_empty_created_at,pct_empty_closed_at,pct_empty_pipeline_source,pct_empty_influenced_by_campaign
500,500,257,51.4,0.0,362,72.39999999999999,0.0,0.0,0.0,0.0,50.4,0.0,0.0


#####Summary
- Contacts
  - 6500 records
  - Keys are unique
  - 25% are missing company_id
  - ~5% of emails are invalid
  - ~6% of emails are duplicates (268 total)
  - 10% of job_title are empty
- Company
  - 1000 records
  - Keys are unique
  - 10% are missing their domain
  - ~6% have a duplicate domain (54 total)
  - 22% are missing industry
  - 24% are missing company_size_band
- Campaign Touchpoints
  - 15000 records
  - Keys are unique
  - 3258 unique contacts had a touchpoint
- Deals
  - 500 records
  - keys are unique
  - 257 unique companies have deals

### Section 2: Relational Integrity Checks

This section is a quick check of each table that contains foreign keys to ensure the foreign key maps to a valid record in the matching table.

In [0]:
select
  *
from information_schema.columns
where table_name in ('crm_raw_contacts','crm_raw_companies','crm_raw_deals','crm_raw_campaign_touchpoints') and column_name like '%_id'

table_catalog,table_schema,table_name,column_name,ordinal_position,column_default,is_nullable,full_data_type,data_type,character_maximum_length,character_octet_length,numeric_precision,numeric_precision_radix,numeric_scale,datetime_precision,interval_type,interval_precision,maximum_cardinality,is_identity,identity_generation,identity_start,identity_increment,identity_maximum,identity_minimum,identity_cycle,is_generated,generation_expression,is_system_time_period_start,is_system_time_period_end,system_time_period_timestamp_generation,is_updatable,partition_index,comment
workspace,default,crm_raw_campaign_touchpoints,touchpoint_id,0,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_campaign_touchpoints,contact_id,1,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_companies,company_id,0,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_contacts,contact_id,0,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_contacts,company_id,1,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_deals,deal_id,0,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_deals,company_id,1,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
workspace,default,crm_raw_deals,primary_contact_id,2,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null


In [0]:
with contact_company_check as(
  select
  n.contact_id,
  n.company_id,
  m.company_id,
  case
    when n.company_id is not null and m.company_id is null then 1
    else 0
    end as missing_company_error
from crm_raw_contacts n
left join crm_raw_companies m
on n.company_id = m.company_id
)

select
  sum(missing_company_error) as total_missing_company_error
from contact_company_check

total_missing_company_error
0


In [0]:
with touchpoint_contact_check as(
  select
  t.touchpoint_id,
  t.contact_id,
  c.contact_id,
  case
    when t.contact_id is not null and c.contact_id is null then 1
    else 0
    end as missing_contact_error
from crm_raw_campaign_touchpoints t
left join crm_raw_contacts c
on t.contact_id = c.contact_id
)

select
  sum(missing_contact_error) as total_missing_contact_error
from touchpoint_contact_check

total_missing_contact_error
0


In [0]:
with deal_contact_check as(
  select
  d.deal_id,
  d.primary_contact_id,
  c.contact_id,
  case
    when d.primary_contact_id is not null and c.contact_id is null then 1
    else 0
    end as missing_contact_error
from crm_raw_deals d
left join crm_raw_contacts c
on d.primary_contact_id = c.contact_id
)

select
  sum(missing_contact_error) as total_missing_contact_error
from deal_contact_check

total_missing_contact_error
0


In [0]:
with deal_company_check as(
  select
  d.deal_id,
  d.company_id,
  c.company_id,
  case
    when d.company_id is not null and c.company_id is null then 1
    else 0
    end as missing_company_error
from crm_raw_deals d
left join crm_raw_companies c
on d.company_id = c.company_id
)

select
  sum(missing_company_error) as total_missing_company_error
from deal_company_check

total_missing_company_error
0


#####Summary
- All company_ids on the Contacts table map to a valid Company
- All contact_ids on the Campaign table map to a valid Contact
- All contact_ids on the Deals table map to a valid Contact
- All company_ids on the Deals table map to a valid Company

### Section 3: Data Governance & Cleaning Rules

This section is to define how raw CRM data will be standardized before further use.

####Company Table

#####Country Standardization

Problem:
- Country values may use inconsistent labels.

Solution:
- Map variants to standard country names
- Unknown/null values remain Unknown

Example:
- USA, U.S., US -> United States

#####Industry Standardization

Problem:
- Industry values may be missing or inconsistent.

Solution:
- Map raw variants into controlled categories
- Null values become Unknown

Example:
- renewables, solar, clean energy -> Renewable Energy

#####Enrichment Status

Problems:
- Some records are incomplete.
- Incomplete records may be missing vital context to drive lead scoring

Solution:
- Set enrichment status based on missing required property:
  - Missing company/domain -> Needs Company Data
  - Duplicate/conflict -> Needs Review
  - Complete required properties -> Complete
- Establish lead enrichment process through vendor

#####Company Deduplication

Problems:
- Companies may appear under multiple names/domains.
- Duplicates can cause deal mismatching and limit context for nurturing pathways

Solution:
- Primary match: domain
- Keep oldest created_at as canonical record
- Add canonical_company_id property to reference the canonical record for all duplicates

Example:
- 2 Companies, SolarCo, Solar Co Inc., both with solarco.com as their domain -> one canonical company

####Contact Table

#####Email Standardization

Problems:
- 5% invalid
- high potential for formatting inconsistencies

Solution:
- mark as invalid if missing the '@' or standard domain structure
- exclude invalid emails from workflow eligibility
- trim whitespace
- lowercase all emails

Examples:
- JohnDoe@GMAIL.COM -> johndoe@gmail.com
- johndoe2gmail.com -> invalid


#####Job Title Normalization

Problem:
- Job titles is a messy free-text property

Solution:
- Preserve raw job_title
- Create standardized job_title_category
- Map titles using keyword logic

Examples:
- Job Titles like VP of Finance, CFO, Finance Lead -> Finance Category
- Job Titles like Head of Recruiting, HR Specialist, Benefits Coordinator -> Human Resources

#####Lifecycle Stage Governance

Problem:
- Lifecycle stages may not match timestamps.

Solution:
- Preserve raw lifecycle stage
- Create lifecycle validation property
- Flag out-of-order timestamps
- Do not overwrite suspicious records without review

Example:
- became_sql_at before became_mql_at -> lifecycle_sequence_error = true

#####Workflow Eligibility

Problem:
- Not every contact should enter marketing workflows.

Solution:
- Eligible contacts must have:
  - Valid email
  - Marketable status = Marketable
  - Not disqualified
  - Not duplicated non-canonical record
- Create a new property to track eligibility called is_workflow_eligible

Example:
- Invalid email + Marketable status -> is_workflow_eligible = false

#####Enrichment Status

Problems:
- Some records are incomplete.
- Incomplete records may be missing vital context to drive lead scoring

Solution:
- Set enrichment status based on missing required properties:
  - Missing company/domain -> Needs Company Data
  - Duplicate/conflict -> Needs Review
  - Complete required properties -> Complete
- Establish lead enrichment process through vendor

#####Contact Deduplication

Problems:
- The same person may appear multiple times.
- Duplicates can cause issues with outreach, nurturing strategies, and deliverability

Solution:

- Primary match: same email
- Keep oldest created_at as canonical record
- Flag other records as duplicates
- Preserve duplicate count for reporting
- Add canonical_contact_id property to reference the canonical record for all duplicates

Example:
- Two records with johndoe@gmail.com -> mark newer record as duplicate

####Campaign Touchpoints Table

#####Contact Link Validation

Problem:
- Touchpoints may reference contacts that don’t exist.

Solution:
- Create invalid_contact_reference property
- Join to contacts table
- If no match -> flag as invalid_contact_reference = true
- Do NOT delete (preserve for audit)

Example:
- contact_id in Campaign Touchpoint Table is 12345, but they are not in contacts -> flagged

#####Timestamp Validation

Problem:
- Touchpoints may be logged before the contact exists, or after the contact has already become a customer.
- These records can distort engagement timelines and attribution analysis.

Solution:
- Establish a sane contact-level timestamp timeline:
  - touchpoint_date must be greater than or equal to contact.created_at
  - If became_customer_at is populated, flag touchpoints that occur after became_customer_at
- Create pre_creation_timestamp_flag
- Create post_customer_touchpoint_flag
- Preserve the raw touchpoint record for audit

Example:
- touchpoint_date = 2025-03-01 and contact.created_at = 2025-03-10 -> pre_creation_timestamp_flag = true
- touchpoint_date = 2025-08-01 and became_customer_at = 2025-07-15 -> post_customer_touchpoint_flag = true

#####High Intent Flag Validation

Problem:
- High-intent flag may not align with action.

Solution:
- Set high_intent_flag = true only for:
  - Demo Request
  - Webinar Attended
  - Sequence Reply
- Override incorrect values

Examples:
- Email Opened touchpoint showing high_intent_flag = true -> change to false

####Deals Table

#####Relational Integrity Validation

Problem:
- Deals may reference a company_id or primary_contact_id that does not exist in the corresponding tables.
- This creates broken relationships, leading to inaccurate reporting, failed joins, and unreliable attribution.

Solution:
- Join deals to crm_clean_companies on company_id
- Join deals to crm_clean_contacts on primary_contact_id
- Create validation flags:
  - invalid_company_reference_flag
  - invalid_contact_reference_flag
  - Flag any records where a referenced entity does not exist
- Preserve raw values for audit and debugging

Examples:
- company_id = 12345 not found in companies -> invalid_company_reference_flag = true
- primary_contact_id = 67890 not found in contacts -> invalid_contact_reference_flag = true

#####Close Date Validation

Problem:
- Closed deals may be missing a closed_at date, or open deals may incorrectly have one populated.
- This creates issues with pipeline reporting and revenue timing.

Solution:
- If deal_stage is Closed Won or Closed Lost then closed_at must be populated
- If deal_stage is not a closed stage then closed_at must be null
- Create validation flag closed_date_error
- Flag any records that violate these conditions for audit

Examples:
- deal_stage = Closed Won and closed_at = null -> closed_date_error = true
- deal_stage = Proposal and closed_at = 2025-06-01 -> closed_date_error = true

#####Deal Timing Logic

Problem:
- Deals may be created before a contact reaches SQL stage, which breaks expected sales lifecycle logic and inflates pipeline metrics.

Solution:
- Join deals to primary contact
- Ensure deal.created_at >= contact.became_sql_at
- Create validation flag deal_before_sql_flag
- Flag any deals created before SQL stage for audit
- Preserve raw values (do not overwrite)

Example:
- deal.created_at = 2025-05-01 and became_sql_at = 2025-05-10 -> deal_before_sql_flag = true

#####Deal Names

Problem:
- Deal names may be inconsistent or lacking information useful for pipeline management

Solution:
- Create standardized deal name structure: [Company] - [Created Date] - [Value]
- Replace old names with new standard

Example:
- Fresh Install of Solar Panels - SolarCo -> SolarCo - 04/28/2026 - $1

#####Duplicate Deals

Problem:
- Same deal entered multiple times

Solution:
- Match rules: standardized deal names
- Keep oldest created_at as canonical record
- Flag other records as duplicates
- Preserve duplicate count for reporting
- Add canonical_deal_id property to reference the canonical record for all duplicates

Example:
- 2 records named SolarCo - 04/28/2026 - $1 -> mark newer record as duplicate

## Section 4: Clean Table Creation

Now that we have our Cleaning Rules, we can process the raw tables into clean tables.

#####Company Table

In [0]:
create or replace table crm_clean_companies as

with base as (
  select
    company_id,
    company_name,
    domain,
    industry,
    company_size_band,
    country,
    region,
    target_account_fit,
    annual_revenue_band,
    created_at
  from crm_raw_companies
),

standardized as (
  select
    *,

    lower(trim(domain)) as domain_clean,

    case
      when lower(trim(country)) in ('us', 'u.s.', 'usa', 'united states', 'united states of america', 'unitedstates') then 'United States'
      when lower(trim(country)) in ('uk', 'u.k.', 'britain', 'united kingdom') then 'United Kingdom'
      when lower(trim(country)) in ('ca', 'can', 'canada') then 'Canada'
      when lower(trim(country)) in ('au', 'australia') then 'Australia'
      when lower(trim(country)) in ('de', 'deutschland', 'germany') then 'Germany'
      when lower(trim(country)) in ('fr', 'france') then 'France'
      when lower(trim(country)) in ('es', 'spain') then 'Spain'
      when lower(trim(country)) in ('nl', 'netherlands') then 'Netherlands'
      when country is null or trim(country) = '' then 'Unknown'
      else trim(country)
    end as country_clean,

    case
      when lower(trim(industry)) like '%solar%' or lower(trim(industry)) like '%renewable%' or lower(trim(industry)) like '%clean energy%' or lower(trim(industry)) like '%wind%' or lower(trim(industry)) = 'energy' then 'Renewable Energy'
      when lower(trim(industry)) like '%finance%' or lower(trim(industry)) like '%fintech%' or lower(trim(industry)) like '%bank%' or lower(trim(industry)) in ('financial services', 'financial svcs') then 'Financial Services'
      when lower(trim(industry)) like '%infra%' or lower(trim(industry)) like '%infrastructure%' or lower(trim(industry)) like '%project finance infrastructure%' or lower(trim(industry)) = 'infrastructure & development' then 'Infrastructure'
      when lower(trim(industry)) like '%gov%' or lower(trim(industry)) like '%public sector%' or lower(trim(industry)) in ('municipal', 'government') then 'Government'
      when lower(trim(industry)) like '%tech%' or lower(trim(industry)) like '%software%' or lower(trim(industry)) = 'saas' then 'Technology'
      when industry is null or trim(industry) = '' then 'Unknown'
      else trim(industry)
    end as industry_clean

  from base
),

deduped as (
  select
    *,

    case
      when domain_clean is not null and count(*) over (partition by domain_clean) > 1 then 1
      else 0
    end as duplicate_company_flag,

    first_value(company_id) over (partition by domain_clean order by created_at asc ) as canonical_company_id

  from standardized
),

final as (
  select
    *,

    case
      when domain_clean is null then 'Missing Domain'
      when industry_clean = 'Unknown' then 'Needs Industry'
      when duplicate_company_flag = 1 then 'Needs Review'
      else 'Complete'
    end as enrichment_status

  from deduped
)

select
  company_id,
  company_name,
  domain,
  industry,
  country,
  domain_clean,
  industry_clean,
  country_clean,
  company_size_band,
  region,
  target_account_fit,
  annual_revenue_band,
  created_at,
  canonical_company_id,
  duplicate_company_flag,
  enrichment_status
from final

num_affected_rows,num_inserted_rows


In [0]:
select
  count(*) as total_rows,
  count(distinct company_id) as unique_company_ids,
  sum(case when domain_clean is null then 1 else 0 end) as missing_domain_clean,
  sum(case when country_clean = 'Unknown' then 1 else 0 end) as unknown_country,
  sum(case when industry_clean = 'Unknown' then 1 else 0 end) as unknown_industry,
  sum(case when canonical_company_id is null then 1 else 0 end) as missing_canonical_company_id,
  sum(case when duplicate_company_flag = 1 then 1 else 0 end) as duplicate_company_records
from crm_clean_companies

total_rows,unique_company_ids,missing_domain_clean,unknown_country,unknown_industry,missing_canonical_company_id,duplicate_company_records
1000,1000,100,0,220,0,110


In [0]:
select
  'country_clean' as category,
  country_clean as value,
  count(*) as total
from crm_clean_companies
group by country_clean

union all

select
  'industry_clean' as category,
  industry_clean as value,
  count(*) as total
from crm_clean_companies
group by industry_clean

order by category, total desc

category,value,total
country_clean,United States,709
country_clean,Canada,84
country_clean,United Kingdom,71
country_clean,Germany,35
country_clean,Netherlands,28
country_clean,Australia,27
country_clean,France,25
country_clean,Spain,21
industry_clean,Unknown,220
industry_clean,Renewable Energy,199


###Contacts Table

In [0]:
create or replace table crm_clean_contacts as

with base as (
  select
    contact_id,
    company_id,
    email,
    first_name,
    last_name,
    job_title,
    job_title_category,
    lifecycle_stage,
    lead_source,
    email_validity_status,
    marketable_status,
    lead_score,
    created_at,
    became_lead_at,
    became_mql_at,
    became_sql_at,
    became_opportunity_at,
    became_customer_at,
    owner_team
  from crm_raw_contacts
),

standardized as (
  select
    *,

    lower(trim(email)) as email_clean,

    case
      when email is null or trim(email) = '' then 0
      when lower(trim(email)) not like '%@%.%' then 0
      when lower(trim(email)) like '%.comm' then 0
      else 1
    end as valid_email_flag,

    case
      when job_title is null or trim(job_title) = '' then 'Unknown'
      when lower(trim(job_title)) like '%ceo%' or lower(trim(job_title)) like '%cfo%' or lower(trim(job_title)) like '%coo%' or lower(trim(job_title)) like '%president%' or lower(trim(job_title)) like '%founder%' or lower(trim(job_title)) like '%chief%' then 'Executive'
      when lower(trim(job_title)) like '%finance%' or lower(trim(job_title)) like '%controller%' or lower(trim(job_title)) like '%cfo%' then 'Finance'
      when lower(trim(job_title)) like '%tax%' or lower(trim(job_title)) like '%counsel%' or lower(trim(job_title)) like '%legal%' then 'Legal'
      when lower(trim(job_title)) like '%energy%' or lower(trim(job_title)) like '%solar%' or lower(trim(job_title)) like '%renewable%' then 'Energy'
      when lower(trim(job_title)) like '%revops%' then 'RevOps'
      when lower(trim(job_title)) like '%sales%' or lower(trim(job_title)) like '%marketing%' then 'Sales/Marketing'
      when lower(trim(job_title)) like '%operations%' or lower(trim(job_title)) like '%ops%' then 'Operations'
      else 'Other'
    end as job_title_category_clean

  from base
),

deduped as (
  select
    *,

    case
      when email_clean is not null and count(*) over (partition by email_clean) > 1 then 1
      else 0
    end as duplicate_contact_flag,

    first_value(contact_id) over (partition by email_clean order by created_at asc) as canonical_contact_id

  from standardized
),

company_mapped as (
  select
    d.*,
    c.canonical_company_id,

    case
      when d.company_id is not null and c.company_id is null then 1
      else 0
    end as invalid_company_reference_flag

  from deduped d
  left join crm_clean_companies c
    on d.company_id = c.company_id
),

validated as (
  select
    *,

    case
      when became_lead_at is not null and became_lead_at < created_at then 1
      when became_mql_at is not null and became_lead_at is not null and became_mql_at < became_lead_at then 1
      when became_sql_at is not null and became_mql_at is not null and became_sql_at < became_mql_at then 1
      when became_opportunity_at is not null and became_sql_at is not null and became_opportunity_at < became_sql_at then 1
      when became_customer_at is not null and became_opportunity_at is not null and became_customer_at < became_opportunity_at then 1
      else 0
    end as lifecycle_timestamp_error,

    case
      when lifecycle_stage = 'MQL' and became_mql_at is null then 1
      when lifecycle_stage = 'SQL' and became_sql_at is null then 1
      when lifecycle_stage = 'Opportunity' and became_opportunity_at is null then 1
      when lifecycle_stage = 'Customer' and became_customer_at is null then 1
      else 0
    end as lifecycle_stage_mismatch_flag,

    case
      when valid_email_flag = 1
        and marketable_status = 'Marketable'
        and lifecycle_stage <> 'Disqualified'
        and contact_id = canonical_contact_id
        then 1
      else 0
    end as is_workflow_eligible

  from company_mapped
),

final as (
  select
    *,

    case
      when valid_email_flag = 0 then 'Needs Contact Data'
      when company_id is null or canonical_company_id is null then 'Needs Company Data'
      when duplicate_contact_flag = 1 and contact_id <> canonical_contact_id then 'Needs Review'
      when job_title_category_clean = 'Unknown' then 'Needs Contact Data'
      else 'Complete'
    end as enrichment_status

  from validated
)

select
  contact_id,
  company_id,
  canonical_company_id,
  email,
  email_clean,
  first_name,
  last_name,
  job_title,
  job_title_category,
  job_title_category_clean,
  lifecycle_stage,
  lead_source,
  email_validity_status,
  marketable_status,
  lead_score,
  created_at,
  became_lead_at,
  became_mql_at,
  became_sql_at,
  became_opportunity_at,
  became_customer_at,
  owner_team,
  canonical_contact_id,
  duplicate_contact_flag,
  valid_email_flag,
  invalid_company_reference_flag,
  lifecycle_timestamp_error,
  lifecycle_stage_mismatch_flag,
  is_workflow_eligible,
  enrichment_status
from final

num_affected_rows,num_inserted_rows


In [0]:
select
  count(*) as total_rows,
  count(distinct contact_id) as unique_contacts,
  count(distinct canonical_contact_id) as canonical_contacts,
  sum(duplicate_contact_flag) as duplicate_records
from crm_clean_contacts

total_rows,unique_contacts,canonical_contacts,duplicate_records
6500,6500,6081,687


In [0]:
select 
  job_title_category_clean,
  count(*) 
from crm_clean_contacts
group by job_title_category_clean
order by count(*) desc

job_title_category_clean,count(*)
Executive,1290
Energy,1122
Other,986
Finance,875
Unknown,650
Operations,579
Legal,548
Sales/Marketing,262
RevOps,188


###Campaign Touchpoints Table

In [0]:
create or replace table crm_clean_campaign_touchpoints as

with base as (
  select
    touchpoint_id,
    contact_id,
    campaign_name,
    campaign_type,
    touchpoint_date,
    touchpoint_action,
    utm_source,
    high_intent_flag
  from crm_raw_campaign_touchpoints
),

contact_mapped as (
  select
    b.*,
    c.created_at as contact_created_at,
    c.became_customer_at,

    case
      when c.contact_id is null then 1
      else 0
    end as invalid_contact_reference_flag

  from base b
  left join crm_clean_contacts c
    on b.contact_id = c.contact_id
),

validated as (
  select
    *,

    case
      when touchpoint_date < contact_created_at then 1
      else 0
    end as pre_creation_timestamp_flag,

    case
      when became_customer_at is not null and touchpoint_date > became_customer_at then 1
      else 0
    end as post_customer_touchpoint_flag,

    case
      when lower(trim(touchpoint_action)) in ('demo request', 'webinar attended', 'sequence reply') then 1
      else 0
    end as high_intent_flag_clean

  from contact_mapped
)

select
  touchpoint_id,
  contact_id,
  campaign_name,
  campaign_type,
  touchpoint_date,
  touchpoint_action,
  utm_source,
  high_intent_flag,
  high_intent_flag_clean,
  invalid_contact_reference_flag,
  pre_creation_timestamp_flag,
  post_customer_touchpoint_flag

from validated

num_affected_rows,num_inserted_rows


In [0]:
select
  count(*) as total,
  sum(invalid_contact_reference_flag) as invalid_contacts,
  sum(invalid_contact_reference_flag) as timestamp_errors,
  sum(post_customer_touchpoint_flag) as post_customer_events
from crm_clean_campaign_touchpoints;

total,invalid_contacts,timestamp_errors,post_customer_events
15000,0,0,230


In [0]:
select
  touchpoint_action,
  high_intent_flag,
  high_intent_flag_clean,
  count(*)
from crm_clean_campaign_touchpoints
group by touchpoint_action, high_intent_flag, high_intent_flag_clean
order by count(*) desc;

touchpoint_action,high_intent_flag,high_intent_flag_clean,count(*)
Form Submit,false,0,4296
Email Click,false,0,2528
Demo Request,true,1,1830
Webinar Attended,true,1,1437
Email Open,false,0,1330
Sequence Reply,true,1,1296
Webinar Registered,false,0,1216
Meeting Booked,true,1,556
Booth Scan,false,0,511


###Deals Table

In [0]:
create or replace table crm_clean_deals as

with base as (
  select
    deal_id,
    company_id,
    primary_contact_id,
    deal_stage,
    amount,
    created_at,
    closed_at,
    pipeline_source,
    influenced_by_campaign
  from crm_raw_deals
),

mapped as (
  select
    b.*,
    c.company_name,
    c.canonical_company_id,
    ct.canonical_contact_id,
    ct.became_sql_at,

    case
      when c.company_id is null then 1
      else 0
    end as invalid_company_reference_flag,

    case
      when ct.contact_id is null then 1
      else 0
    end as invalid_contact_reference_flag

  from base b
  left join crm_clean_companies c
    on b.company_id = c.company_id
  left join crm_clean_contacts ct
    on b.primary_contact_id = ct.contact_id
),

standardized as (
  select
    *,

    concat(
      company_name,
      ' - ',
      date_format(created_at, 'yyyy-MM-dd'),
      ' - $',
      cast(round(amount, 0) as int)
    ) as deal_name_clean,

    case
      when deal_stage in ('Closed Won', 'Closed Lost') and closed_at is null then 1
      when deal_stage not in ('Closed Won', 'Closed Lost') and closed_at is not null then 1
      else 0
    end as closed_date_error,

    case
      when became_sql_at is not null and created_at < became_sql_at then 1
      else 0
    end as deal_before_sql_flag

  from mapped
),

deduped as (
  select
    *,

    case
      when count(*) over (partition by deal_name_clean) > 1 then 1
      else 0
    end as duplicate_deal_flag,

    first_value(deal_id) over (
      partition by deal_name_clean
      order by created_at asc
    ) as canonical_deal_id

  from standardized
)

select
  deal_id,
  canonical_deal_id,
  company_id,
  canonical_company_id,
  primary_contact_id,
  canonical_contact_id,
  deal_name_clean,
  deal_stage,
  amount,
  created_at,
  closed_at,
  pipeline_source,
  influenced_by_campaign,
  closed_date_error,
  deal_before_sql_flag,
  duplicate_deal_flag,
  invalid_company_reference_flag,
  invalid_contact_reference_flag

from deduped

num_affected_rows,num_inserted_rows


In [0]:
select
  count(*) as total,
  count(distinct deal_id) as unique_deals,
  count(distinct canonical_deal_id) as canonical_deals,
  sum(invalid_company_reference_flag) as invalid_company_references,
  sum(invalid_contact_reference_flag) as invalid_contact_references,
  sum(closed_date_error) as closed_date_errors,
  sum(deal_before_sql_flag) as deal_before_sql_errors
from crm_clean_deals

total,unique_deals,canonical_deals,invalid_company_references,invalid_contact_references,closed_date_errors,deal_before_sql_errors
500,500,500,0,0,1,28


In [0]:
select
  deal_stage,
  count(*) as total,
  sum(closed_date_error) as closed_date_errors
from crm_clean_deals
group by deal_stage
order by total desc

deal_stage,total,closed_date_errors
Closed Lost,193,0
Discovery,75,0
Proposal,65,0
Negotiation,60,0
Closed Won,56,1
Created,51,0
